# 公開用注記（原実験Notebook）

このNotebookは開発時のコードを記録として公開したものです。実験ロジックは変更していません。

**実行上の依存関係：** 開発時のGoogle Driveにある歩行モデル・設定・上位モデルを参照します。最終公開モデルのみでの単独実行は未確認です。

公開版での一連の再実行は確認していません。実行時は参照パスや必要ファイルを確認してください。


# Phase45：Genesisで12D上位PPOを動かす短時間ロールアウト

Phase40の80ステッププローブを流用し、Phase42の12D初期モデルを追跡・回避双方で使用。下位歩行PPOは固定、再学習なし。行動前12D観測と行動後のtrajectoryを記録。報酬の時刻整合性はCSV受領後に監査する。GPUが必要。

In [ ]:
import sys, subprocess
subprocess.run([sys.executable,"-m","pip","install","-q","genesis-world==1.4.4","rsl-rl-lib==5.5.1"],check=True)
import importlib.metadata as im
import torch
print("Python:",sys.version.split()[0])
print("Genesis:",im.version("genesis-world"))
print("rsl-rl-lib:",im.version("rsl-rl-lib"))
print("CUDA available:",torch.cuda.is_available())
if torch.cuda.is_available(): print("GPU:",torch.cuda.get_device_name(0))
assert torch.cuda.is_available(),"GPUランタイムを選択してください"

In [ ]:
import subprocess
from pathlib import Path
root=Path("/content/genesis_official")
commit="105898fc0a5f068f22dffbbfa39f5ea89be42a1e"
if not (root/".git").exists():
    subprocess.run(["git","clone","--filter=blob:none","https://github.com/Genesis-Embodied-AI/genesis-world.git",str(root)],check=True)
subprocess.run(["git","-C",str(root),"fetch","--depth","1","origin",commit],check=True)
subprocess.run(["git","-C",str(root),"checkout","--detach",commit],check=True)
source_dir=root/"examples"/"locomotion"
assert (source_dir/"go2_env.py").is_file(),"go2_env.pyが見つかりません"
print("SOURCE OK:",source_dir/"go2_env.py")
print("COMMIT:",subprocess.check_output(["git","-C",str(root),"rev-parse","HEAD"],text=True).strip())

In [ ]:
from google.colab import drive
from pathlib import Path
import uuid,json,torch,pickle,hashlib
if not Path("/content/drive/MyDrive").exists(): drive.mount("/content/drive")
root=Path("/content/drive/MyDrive/PhysicalAI_Go2_Pursuit")
checkpoints={"phase112":root/"training/locomotion/phase112_78e8a86701/phase112_final.pt","phase16":root/"training/locomotion/phase16_generalize_2ee1ca397e/phase16_final.pt"}
candidates=sorted((root/"training/pursuit").glob("pursuit_v045_*/upper_pursuit_ppo.pt"),key=lambda p:p.stat().st_mtime,reverse=True)
assert candidates,"07の再学習モデルが見つかりません"
upper_checkpoint=candidates[0]
print("今回の追跡モデル:",upper_checkpoint)
evasion_checkpoint=root/"training/evasion/evasion_995dbb39/upper_evasion_ppo.pt"
cfg_paths={"phase112":root/"training/locomotion/locomotion_51ec8dd43f/planned_cfgs.pkl","phase16":root/"training/locomotion/phase16_generalize_2ee1ca397e/phase16_cfgs.pkl"}
cfg_path=cfg_paths["phase16"]
for label,p in {**checkpoints,"pursuit":upper_checkpoint,"evasion":evasion_checkpoint,**{"cfg_"+k:v for k,v in cfg_paths.items()}}.items():
    assert p.is_file(),f"必要ファイルがありません {label}: {p}"
    print("確認:",label,p.name,p.stat().st_size,"bytes")
assert torch.cuda.is_available(),"GPUランタイムが必要です"
print("GPU:",torch.cuda.get_device_name(0))
with cfg_path.open("rb") as f:
    cfgs=pickle.load(f)
assert isinstance(cfgs,(tuple,list)) and len(cfgs)==5,"設定ファイル形式が想定外です"
env_cfg,obs_cfg,reward_cfg,cmd_cfg,train_cfg=cfgs
assert len(env_cfg["joint_names"])==12,"関節設定が12個ではありません"
locomotion_ck=torch.load(checkpoints["phase16"],map_location="cpu",weights_only=False)
assert isinstance(locomotion_ck,dict),"歩行モデル形式が想定外です"
print("歩行チェックポイントキー:",list(locomotion_ck))
for label,p in (("追跡",upper_checkpoint),("回避",evasion_checkpoint)):
    ck=torch.load(p,map_location="cpu",weights_only=False)
    assert "actor_critic" in ck, f"{label}のモデル形式が想定外です"
    print(label,"上位PPOキー確認OK")
out_root=root/"evaluation/genesis";out_root.mkdir(parents=True,exist_ok=True)
print("事前ファイル確認完了。次はスクリプト構造の検証です。")
folders=sorted((root/"training/upper12_transfer").glob("phase42_*"),key=lambda p:p.stat().st_mtime,reverse=True)
folder=next((p for p in folders if all((p/f"upper_{role}_12d_initialized.pt").is_file() for role in ("pursuit","evasion"))),None)
assert folder is not None,"Phase42 12D初期モデルが見つかりません"
upper_checkpoint=folder/"upper_pursuit_12d_initialized.pt"
evasion_checkpoint=folder/"upper_evasion_12d_initialized.pt"
for p in (upper_checkpoint,evasion_checkpoint):
 ck=torch.load(p,map_location="cpu",weights_only=False)
 assert ck["config"]["input_dim"]==12 and ck["config"]["trained_on_12d"] is False
 print("12D",p)


In [ ]:
import ast
script="import torch,pickle,csv,math,sys\nfrom pathlib import Path\noutput_dir=Path(__file__).resolve().parent\ncase=sys.argv[1]\nassert len(sys.argv)==14,\"usage: script.py case kp kd max_steps mode checkpoint yaw_bias upper_checkpoint evasion_checkpoint speed_mode test_pursuer_speed cfg_path\"\nspeed_mode=sys.argv[10]\ntest_pursuer_speed=float(sys.argv[11])\ntest_yaw=float(sys.argv[12])\nassert speed_mode in (\"tag\",\"rule_tag\",\"asymmetric\")\nmax_steps=int(sys.argv[4]);mode=sys.argv[5]\nassert max_steps==3000 and mode==\"mutual_tag\"\nkp=float(sys.argv[2]);kd=float(sys.argv[3])\nyaw_bias=float(sys.argv[7])\nassert 0<kp<=100 and 0<kd<=10\nupper_checkpoint=Path(sys.argv[8])\nevasion_checkpoint=Path(sys.argv[9])\nplacements={\n    \"phase75_reference\":((-0.8,-0.4),(0.9,0.5),(20,-50)),\n    \"close_start\":((-0.55,-0.15),(0.55,0.15),(0,0)),\n    \"head_on\":((-0.9,0.0),(0.9,0.0),(0,0)),\n    \"diagonal\":((-0.8,-0.7),(0.8,0.7),(35,-40)),\n    \"sideways\":((-0.9,0.0),(0.9,0.0),(90,90)),\n    \"opposite\":((-0.9,0.0),(0.9,0.0),(180,180)),\n}\nplacement,policy_kind=case.rsplit(\"__\",1)\nassert placement in placements and policy_kind in (\"fixed\",\"swap5\",\"tag\",\"rule_tag\"),case\na_xy,b_xy,offsets=placements[placement]\ninitial_xy=(a_xy,b_xy)\nheading_offsets={case:(math.radians(offsets[0]),math.radians(offsets[1]))}\n\ndef face_angle(a,b):\n    return math.atan2(b[1]-a[1],b[0]-a[0])\n# A heads toward B; B heads away from A.\ninitial_yaws=(face_angle(initial_xy[0],initial_xy[1])+heading_offsets[case][0],\n              face_angle(initial_xy[1],initial_xy[0])+math.pi+heading_offsets[case][1])\nimport genesis as gs\nfrom genesis.vis.rasterizer import Rasterizer\nfrom genesis.utils.geom import inv_quat,transform_by_quat\nRasterizer.build=lambda self:None\ncfg_path=Path(sys.argv[13])\nwith cfg_path.open(\"rb\") as f:\n    env,obs_cfg,reward,cmd,train=pickle.load(f)\ncheckpoint_path=Path(sys.argv[6])\nassert checkpoint_path.is_file(),f\"Checkpoint not found: {checkpoint_path}\"\ncheckpoint=torch.load(checkpoint_path,map_location=\"cpu\",weights_only=False)\nprint(\"CHECKPOINT TYPE\",type(checkpoint).__name__,flush=True)\nprint(\"CHECKPOINT KEYS\",list(checkpoint.keys()) if isinstance(checkpoint,dict) else \"not a dict\",flush=True)\nweights=checkpoint.get(\"actor_state_dict\",checkpoint.get(\"model_state_dict\",checkpoint.get(\"model\",checkpoint.get(\"state_dict\")))) if isinstance(checkpoint,dict) else None\nassert isinstance(weights,dict),\"Unknown checkpoint layout: see CHECKPOINT KEYS above\"\nprint(\"STATE DICT KEYS SAMPLE\",list(weights)[:25],flush=True)\nkeys=sorted((k for k in weights if k.endswith(\".weight\") and (k.startswith(\"mlp.\") or k.startswith(\"actor.\") or k.split(\".\")[0].isdigit())),key=lambda k:int(k.split(\".\")[-2]))\nassert keys and weights[keys[0]].shape[1]==45 and weights[keys[-1]].shape[0]==12\nprint(\"TRAIN CFG TYPE\",type(train).__name__,flush=True)\nprint(\"TRAIN CFG KEYS\",list(train.keys()) if isinstance(train,dict) else repr(train),flush=True)\nprint(\"ACTOR LAYER SHAPES\",[(k,tuple(weights[k].shape)) for k in keys],flush=True)\n# Genesis Go2 PPO default is ELU; verify model dimensions before physics startup.\nassert len(keys)==4 and weights[keys[0]].shape[1]==45 and weights[keys[-1]].shape[0]==12,\"Unexpected actor architecture\"\nlayers=[]\nfor i,k in enumerate(keys):\n    w=weights[k];l=torch.nn.Linear(w.shape[1],w.shape[0]);l.weight.data.copy_(w);l.bias.data.copy_(weights[k.replace(\".weight\",\".bias\")]);layers.append(l)\n    if i<len(keys)-1:layers.append(torch.nn.ELU())\ngs.init(backend=gs.gpu,precision=\"32\",logging_level=\"warning\",seed=1,performance_mode=False)\npolicy=torch.nn.Sequential(*layers).to(gs.device).eval()\n# Phase74 upper PPO actor: only 2D pilot training, no Genesis fine-tuning.\nclass UpperActor(torch.nn.Module):\n    def __init__(self,max_forward):\n        super().__init__()\n        self.max_forward=max_forward\n        self.actor=torch.nn.Sequential(torch.nn.Linear(8,128),torch.nn.Tanh(),torch.nn.Linear(128,128),torch.nn.Tanh(),torch.nn.Linear(128,2))\n    def forward(self,x):\n        raw=self.actor(x)\n        return torch.stack((torch.sigmoid(raw[:,0])*self.max_forward,torch.tanh(raw[:,1])),1)\nupper=UpperActor(0.45).to(gs.device).eval()\nck=torch.load(upper_checkpoint,map_location=\"cpu\",weights_only=False)\nstate_upper=ck[\"actor_critic\"]\nupper.load_state_dict({k:v for k,v in state_upper.items() if k.startswith(\"actor.\")},strict=True)\nprint(\"UPPER CHECKPOINT\",upper_checkpoint,\"case\",case,flush=True)\nevader_upper=UpperActor(0.30).to(gs.device).eval()\nevasion_ck=torch.load(evasion_checkpoint,map_location=\"cpu\",weights_only=False)\nevader_upper.load_state_dict({k:v for k,v in evasion_ck[\"actor_critic\"].items() if k.startswith(\"actor.\")},strict=True)\nprint(\"EVASION CHECKPOINT\",evasion_checkpoint,flush=True)\n\nscene=gs.Scene(sim_options=gs.options.SimOptions(dt=0.02,substeps=2),rigid_options=gs.options.RigidOptions(enable_self_collision=False,max_collision_pairs=40,tolerance=1e-5),show_viewer=False)\nscene.add_entity(gs.morphs.URDF(file=\"urdf/plane/plane.urdf\",fixed=True))\nrobots=[scene.add_entity(gs.morphs.URDF(file=\"urdf/go2/urdf/go2.urdf\",pos=(xy[0],xy[1],0.42))) for xy in initial_xy]\nscene.build()\nindices=[[int(r.get_joint(n).dofs_idx_local[0]) for n in env[\"joint_names\"]] for r in robots]\nassert indices[0]==indices[1]==[9,10,11,6,7,8,15,16,17,12,13,14],indices\ndefault=torch.tensor([env[\"default_joint_angles\"][n] for n in env[\"joint_names\"]],device=gs.device)\ngravity=torch.tensor([0.,0.,-1.],device=gs.device)\nsc=obs_cfg[\"obs_scales\"]\ncommands=torch.tensor([[0.3,0.,0.],[0.3,0.,0.]],device=gs.device)\nheading_gain=1.0\nyaw_limit=0.8\ncommand_scale=torch.tensor([sc[\"lin_vel\"],sc[\"lin_vel\"],sc[\"ang_vel\"]],device=gs.device)\nactions=torch.zeros((2,12),device=gs.device);last=torch.zeros_like(actions)\nfor i,r in enumerate(robots):\n    r.set_dofs_kp([kp]*12,indices[i]);r.set_dofs_kv([kd]*12,indices[i])\n    yaw0=initial_yaws[i]\n    q=[initial_xy[i][0],initial_xy[i][1],0.42,math.cos(yaw0/2),0.,0.,math.sin(yaw0/2)]+[env[\"default_joint_angles\"][j.name] for j in r.joints[1:]]\n    r.set_qpos(torch.tensor(q,device=gs.device),zero_velocity=True,skip_forward=False)\n# Confirm initial world-frame orientation before the first control step.\nfor i,r in enumerate(robots):\n    pos=r.get_pos().reshape(-1)[:3]\n    quat=r.get_quat().reshape(-1)[:4]\n    w,x,y,z=quat.detach().cpu().tolist()\n    yaw=math.atan2(2*(w*z+x*y),1-2*(y*y+z*z))\n    target=robots[1-i].get_pos().reshape(-1)[:3]\n    delta=target[:2]-pos[:2]\n    desired=math.atan2(float(delta[1]),float(delta[0]))\n    err=math.atan2(math.sin(desired-yaw),math.cos(desired-yaw))\n    expected=initial_yaws[i]\n    print(\"INITIAL\", \"AB\"[i], \"yaw\",yaw,\"expected\",expected,\"heading_error\",err,\"pos\",pos.detach().cpu().tolist(),flush=True)\n    assert abs(math.atan2(math.sin(yaw-expected),math.cos(yaw-expected)))<0.02, \"Initial yaw not reflected before first step\"\nprint(\"READY: two Go2, shared scene, policy 45->12, CASE\",case,\"KP\",kp,\"KD\",kd,flush=True)\nprint(\"CONFIG\",{\"kp\":env[\"kp\"],\"kd\":env[\"kd\"],\"action_scale\":env[\"action_scale\"],\"latency\":env[\"simulate_action_latency\"],\"obs_scales\":sc,\"actor\":train.get(\"actor\")},flush=True)\ndef state(r,idx):\n    p=r.get_pos().reshape(-1)[:3];q=r.get_quat().reshape(-1)[:4];inv=inv_quat(q)\n    return p,q,transform_by_quat(r.get_ang().reshape(-1)[:3],inv),transform_by_quat(gravity,inv),r.get_dofs_position(idx).reshape(-1)[:12],r.get_dofs_velocity(idx).reshape(-1)[:12]\nprev_positions=None\nprev_vel=torch.zeros((2,2),device=gs.device)\nprev_yaws=None\nprev_omega=torch.zeros(2,device=gs.device)\nrows=[]\nroles=[]\ndiag=[]\npose_rows=[]\njoint_target_rows=[]\npose_length=None\ncalibration_input=[-0.8,-0.4,-0.2,0.0,0.2,0.4,0.8]\ncalibration_output=[-0.355,-0.192,-0.122,-0.058,0.004,0.075,0.212]\ndef inverse_yaw(target):\n    target=max(calibration_output[0],min(calibration_output[-1],target))\n    for j in range(len(calibration_output)-1):\n        lo,hi=calibration_output[j:j+2]\n        if lo<=target<=hi:\n            t=(target-lo)/(hi-lo)\n            return calibration_input[j]+t*(calibration_input[j+1]-calibration_input[j])\n    raise RuntimeError(\"Calibration interpolation failed\")\nswitched=False\nlast_tag_step=-100000\nescape_hold_yaw=None\ntag_armed=True\ntag_events=[]\nwith torch.no_grad():\n for step in range(1,max_steps+1):\n    states=[state(r,indices[i]) for i,r in enumerate(robots)]\n    current_xy=torch.stack([z[0][:2] for z in states])\n    current_yaws=[]\n    for z in states:\n        w,x,y,qz=z[1].unbind()\n        current_yaws.append(torch.atan2(2*(w*qz+x*y),1-2*(y*y+qz*qz)))\n    current_yaws=torch.stack(current_yaws)\n    if prev_positions is not None:\n        vel_world=(current_xy-prev_positions)/0.02\n        omega=torch.atan2(torch.sin(current_yaws-prev_yaws),torch.cos(current_yaws-prev_yaws))/0.02\n        # Smooth noisy finite differences; initialize from rest.\n        prev_vel=0.8*prev_vel+0.2*vel_world\n        prev_omega=0.8*prev_omega+0.2*omega\n    prev_positions=current_xy.clone()\n    prev_yaws=current_yaws.clone()\n    headings=[];errors=[]\n    delta=current_xy[1]-current_xy[0]\n    distance_now=float(torch.linalg.norm(current_xy[1]-current_xy[0]))\n    if policy_kind in (\"tag\",\"rule_tag\"):\n        if (not tag_armed) and distance_now>=0.85:\n            tag_armed=True\n        if tag_armed and distance_now<=0.8 and step-last_tag_step>=150:\n            switched=not switched\n            last_tag_step=step\n            escape_hold_yaw=None  # PPO chooses evasion heading after tag\n            tag_armed=False\n            tag_events.append([step,step*0.02,distance_now,\"B\" if switched else \"A\"])\n            print(\"TAG\",step,\"time\",step*0.02,\"distance\",distance_now,flush=True)\n    else:\n        switched=(policy_kind==\"swap5\" and step*0.02>=5.0)\n    for i in range(2):\n        pursuer=(i==1) if switched else (i==0)\n        relative=current_xy[1-i]-current_xy[i]\n        desired=torch.atan2(relative[1],relative[0])\n        if not pursuer:\n            desired=torch.atan2(-relative[1],-relative[0])\n        err=torch.atan2(torch.sin(desired-current_yaws[i]),torch.cos(desired-current_yaws[i]))\n        headings.append(desired);errors.append(err)\n        roles.append([step,step*0.02,\"AB\"[i],\"pursuer\" if pursuer else \"evader\",float(desired),float(err)])\n    def upper_obs(i):\n        j=1-i\n        c=torch.cos(current_yaws[i]);sn=torch.sin(current_yaws[i])\n        relative=current_xy[j]-current_xy[i]\n        relative_vel=prev_vel[j]-prev_vel[i]\n        dyaw=torch.atan2(torch.sin(current_yaws[j]-current_yaws[i]),torch.cos(current_yaws[j]-current_yaws[i]))\n        forward_speed=(prev_vel[i]*torch.stack((c,sn))).sum()\n        return torch.stack(((c*relative[0]+sn*relative[1])/3,\n            (-sn*relative[0]+c*relative[1])/3,\n            c*relative_vel[0]+sn*relative_vel[1],\n            -sn*relative_vel[0]+c*relative_vel[1],\n            torch.sin(dyaw),torch.cos(dyaw),\n            forward_speed,prev_omega[i])).reshape(1,8)\n    raw_forward=[0.0,0.0]\n    raw_yaw=[0.0,0.0]\n    for i in range(2):\n        pursuer=(i==1) if switched else (i==0)\n        role_policy=upper if pursuer else evader_upper\n        role_observation=upper_obs(i)\n        assert role_observation.shape==(1,8)\n        cmd=role_policy(role_observation)[0]\n        # No rule-based override: learned forward and yaw commands drive the locomotion PPO.\n        assert torch.isfinite(cmd).all()\n        assert 0.0<=float(cmd[0])<=(0.45001 if pursuer else 0.30001) and abs(float(cmd[1]))<=1.00001\n        raw_forward[i]=float(cmd[0])\n        raw_yaw[i]=float(cmd[1])\n        commands[i,0]=cmd[0]\n        commands[i,1]=0.0\n        commands[i,2]=cmd[1]\n    observations=torch.stack([torch.cat((s[2]*sc[\"ang_vel\"],s[3],commands[i]*command_scale,(s[4]-default)*sc[\"dof_pos\"],s[5]*sc[\"dof_vel\"],actions[i])) for i,s in enumerate(states)])\n    assert observations.shape==(2,45) and torch.isfinite(observations).all()\n    new=policy(observations).clamp(-env[\"clip_actions\"],env[\"clip_actions\"])\n    assert new.shape==(2,12) and torch.isfinite(new).all()\n    applied=last if env[\"simulate_action_latency\"] else new\n    targets=applied*env[\"action_scale\"]+default\n    for i,r in enumerate(robots):\n        order=torch.argsort(torch.tensor(indices[i],device=gs.device))\n        r.control_dofs_position(targets[i,order],slice(6,18))\n    metrics=[]\n    for i in range(2):\n        err=targets[i]-states[i][4]\n        metrics.append((float(torch.sqrt(torch.mean(err**2))),float(torch.max(torch.abs(err))),float(torch.sqrt(torch.mean(states[i][5]**2))),int(torch.count_nonzero(torch.abs(new[i])>=env[\"clip_actions\"]-1e-5)),float(torch.linalg.norm(observations[i]))))\n    scene.step()\n    actions=new.clone();last=new.clone()\n    for i,r in enumerate(robots):\n        p,q,ang,_,_,_=state(r,indices[i])\n        qpos=r.get_qpos().reshape(-1).detach().cpu().tolist()\n        if pose_length is None:pose_length=len(qpos);print('QPOS LENGTH',pose_length,flush=True)\n        assert len(qpos)==pose_length and pose_length==19,(len(qpos),pose_length)\n        pose_rows.append([step,step*0.02,'AB'[i],*qpos])\n        # Target and measured angles are both in env['joint_names'] order.\n        actual_joint=r.get_dofs_position(indices[i]).reshape(-1)[:12]\n        joint_target_rows.append([step,step*0.02,'AB'[i],*[float(v) for v in targets[i].detach().cpu()],*[float(v) for v in actual_joint.detach().cpu()]])\n        w,x,y,z=q.detach().cpu().tolist()\n        yaw=math.atan2(2*(w*z+x*y),1-2*(y*y+z*z))\n        roll=math.degrees(math.atan2(2*(w*x+y*z),1-2*(x*x+y*y)))\n        pitch=math.degrees(math.asin(max(-1,min(1,2*(w*y-z*x)))))\n        distance=float(torch.linalg.norm(robots[0].get_pos().reshape(-1)[:2]-robots[1].get_pos().reshape(-1)[:2]))\n        row=[step,step*0.02,\"AB\"[i],*p.detach().cpu().tolist(),yaw,float(ang[2]),float(new[i].norm()),roll,pitch,float(commands[i,2]),float(errors[i]),distance]\n        assert all(math.isfinite(float(v)) for v in row[3:]),row\n        rows.append(row)\n        actual=r.get_dofs_position(indices[i]).reshape(-1)[:12]\n        m=metrics[i]\n        d=[step,step*0.02,\"AB\"[i],float(p[2]),roll,pitch,yaw,float(commands[i,0]),float(commands[i,2]),raw_yaw[i],float(ang[2]),raw_forward[i],m[0],m[1],float(torch.sqrt(torch.mean((targets[i]-actual)**2))),m[2],m[3],m[4],float(torch.mean(torch.abs(new[i])))]\n        diag.append(d)\n        if step in (1,25,50,100,150,200,250):print(\"STEP\",step,row,\"DIAG\",d,flush=True)\n    if step%50==0:\n        print(\"PROGRESS\",step,\"of\",max_steps,\"distance\",distance,\"min_height\",min(float(robots[j].get_pos().reshape(-1)[2]) for j in range(2)),flush=True)\n    if any(float(robots[j].get_pos().reshape(-1)[2])<0.18 for j in range(2)):\n        print(\"EARLY_STOP_LOW_HEIGHT\",step,flush=True)\n        break\nwith (output_dir/\"tag_events.csv\").open(\"w\",newline=\"\") as f:\n    wr=csv.writer(f);wr.writerow([\"step\",\"time_s\",\"distance_xy_m\",\"new_pursuer\"]);wr.writerows(tag_events)\nprint(\"TAG_COUNT\",len(tag_events),flush=True)\nwith (output_dir/\"roles.csv\").open(\"w\",newline=\"\") as f:\n    wr=csv.writer(f);wr.writerow([\"step\",\"time_s\",\"robot\",\"role\",\"desired_yaw_rad\",\"heading_error_rad\"]);wr.writerows(roles)\nwith (output_dir/\"trajectory.csv\").open(\"w\",newline=\"\") as f:\n    wr=csv.writer(f);wr.writerow([\"step\",\"time_s\",\"robot\",\"x\",\"y\",\"z\",\"yaw\",\"yaw_rate\",\"action_norm\",\"roll_deg\",\"pitch_deg\",\"yaw_cmd\",\"heading_error\",\"distance_xy\"]);wr.writerows(rows)\nwith (output_dir/\"diagnostics.csv\").open(\"w\",newline=\"\") as f:\n    wr=csv.writer(f);wr.writerow([\"step\",\"time_s\",\"robot\",\"height_m\",\"roll_deg\",\"pitch_deg\",\"yaw_rad\",\"forward_cmd\",\"yaw_cmd\",\"raw_upper_yaw_cmd\",\"body_yaw_rate\",\"raw_forward_cmd\",\"joint_target_rmse_before_rad\",\"joint_target_max_error_before_rad\",\"joint_target_rmse_after_rad\",\"joint_velocity_rms_rad_s\",\"clipped_action_count\",\"observation_norm\",\"mean_abs_action\"]);wr.writerows(diag)\nwith (output_dir/\"joint_poses.csv\").open(\"w\",newline=\"\") as f:\n    wr=csv.writer(f);wr.writerow([\"step\",\"time_s\",\"robot\"]+[\"qpos_\"+str(j) for j in range(pose_length)]);wr.writerows(pose_rows)\nwith (output_dir/\"joint_targets_actual.csv\").open(\"w\",newline=\"\") as f:\n    wr=csv.writer(f);wr.writerow([\"step\",\"time_s\",\"robot\"]+[\"target_\"+str(j) for j in range(12)]+[\"actual_\"+str(j) for j in range(12)]);wr.writerows(joint_target_rows)\nwith (output_dir/\"joint_name_mapping.csv\").open(\"w\",newline=\"\") as f:\n    wr=csv.writer(f);wr.writerow([\"index\",\"joint_name\",\"dofs_idx_local\"]);wr.writerows([[j,env[\"joint_names\"][j],indices[0][j]] for j in range(12)])\nprint(\"PHASE86 FINISHED\",case,kp,kd,\"steps\",step,\"mode\",mode,\"qpos_rows\",len(pose_rows),flush=True)\n"
ast.parse(script)
assert "UpperActor(0.45)" in script and "evader_upper=UpperActor(0.30)" in script
assert "commands[i,0]=cmd[0]" in script and "commands[i,2]=cmd[1]" in script
assert "assert max_steps==3000" in script
print("SCRIPT SYNTAX AND CONTROL CHECK OK")

## Phase40専用のログ追加。元の制御方策は変更しません。

In [ ]:
import ast
assert script.count('assert max_steps==3000 and mode=="mutual_tag"')==1
script=script.replace('assert max_steps==3000 and mode=="mutual_tag"','assert max_steps==80 and mode=="mutual_tag"')
needle='    current_xy=torch.stack([z[0][:2] for z in states])'
assert script.count(needle)==1
script=script.replace(needle,'''    phase40_sensor=[]
    for s in states:
        w,x,y,z=s[1].unbind()
        roll=torch.atan2(2*(w*x+y*z),1-2*(x*x+y*y))
        pitch=torch.asin(torch.clamp(2*(w*y-z*x),-1.,1.))
        phase40_sensor.append((roll,pitch,s[2]))
'''+needle,1)
needle='    observations=torch.stack([torch.cat('
assert script.count(needle)==1
script=script.replace(needle,'''    for i in range(2):
        obs8=upper_obs(i)
        roll,pitch,omega_body=phase40_sensor[i]
        obs12=torch.cat((obs8,torch.stack((roll,pitch,omega_body[0],omega_body[1])).reshape(1,4)),dim=1)
        assert obs8.shape==(1,8) and obs12.shape==(1,12) and torch.isfinite(obs12).all()
        q=states[i][1]
        raw_omega=robots[i].get_ang().reshape(-1)[:3]
        phase40_rows.append([step,(step-1)*0.02,"AB"[i],*[float(v) for v in q],
          *[float(v) for v in raw_omega],*[float(v) for v in obs12.reshape(-1)]])
'''+needle,1)
needle='with torch.no_grad():\n for step in range(1,max_steps+1):'
assert script.count(needle)==1
script=script.replace(needle,'phase40_rows=[]\n'+needle,1)
script+='''
with (output_dir/"phase40_pre_action_obs12.csv").open("w",newline="") as f:
    wr=csv.writer(f)
    wr.writerow(["step","pre_action_time_s","robot"]+
      ["quat_w","quat_x","quat_y","quat_z"]+
      ["omega_raw_x","omega_raw_y","omega_raw_z"]+
      ["obs_"+str(j) for j in range(12)])
    wr.writerows(phase40_rows)
print("PHASE40 ROWS",len(phase40_rows),flush=True)
'''
ast.parse(script)
# GPUを使う前に形状を単体テスト
import torch
test8=torch.zeros((1,8))
test4=torch.zeros((1,4))
test12=torch.cat((test8,test4),dim=1)
assert test12.shape==(1,12)
assert "obs12.shape==(1,12)" in script
print("Phase40 shape test PASS: (1,8)+(1,4)->(1,12)")
print("Phase40 modified script parsed. Existing 8D PPO remains unchanged.")
# 12D Actorへの切替。Phase40で記録した行動前観測をそのまま利用する。
assert script.count("torch.nn.Linear(8,128)")==1
script=script.replace("torch.nn.Linear(8,128)","torch.nn.Linear(12,128)",1)
a="    for i in range(2):\n        obs8=upper_obs(i)"
assert script.count(a)==1
script=script.replace(a,"    phase45_obs12=[]\n"+a,1)
a="        assert obs8.shape==(1,8) and obs12.shape==(1,12) and torch.isfinite(obs12).all()"
assert script.count(a)==1
script=script.replace(a,a+"\n        phase45_obs12.append(obs12)",1)
assert script.count("role_observation=upper_obs(i)")==1
script=script.replace("role_observation=upper_obs(i)","role_observation=phase45_obs12[i]",1)
assert script.count("assert role_observation.shape==(1,8)")==1
script=script.replace("assert role_observation.shape==(1,8)","assert role_observation.shape==(1,12)",1)
ast.parse(script)
print("Phase45 12D Actor patch PASS")

# 観測生成ブロックの位置を、行動選択の前へ移す。
# scriptは実際の改行を含むPythonコード文字列である点に注意。
start=script.index("    phase45_obs12=[]" + chr(10))
end=script.index("    observations=torch.stack(",start)
obs_block=script[start:end]
assert "phase45_obs12.append(obs12)" in obs_block
assert "phase40_rows.append(" in obs_block
script=script[:start]+script[end:]
anchor="    raw_forward=[0.0,0.0]" + chr(10)
assert script.count(anchor)==1
script=script.replace(anchor,obs_block+anchor,1)
assert script.index("    phase45_obs12=[]")<script.index("role_observation=phase45_obs12[i]")
assert script.index("phase40_sensor=[]")<script.index("    phase45_obs12=[]")
assert script.index("    phase45_obs12=[]")<script.index("    raw_forward=[0.0,0.0]")
ast.parse(script)
print("Phase45 execution-order preflight PASS: sensor -> obs12 -> actor -> locomotion")


## 80ステップ実行・結果検証・ZIPダウンロード

## Phase46：学習用データ収集への変更
潜在Gaussian行動、old log probability、Critic value、行動後報酬、次状態12D、転倒/役割交代の境界を記録。重み更新なし。

In [ ]:
# Phase46 学習用ロールアウト：Phase45のrunnerを変更し、モデルは更新しない
import ast, json, zipfile, subprocess, sys, time, uuid, hashlib
from pathlib import Path
import pandas as pd, numpy as np
from google.colab import files

def replace_exact(before,after):
 global script
 n=script.count(before)
 assert n==1,f"置換アンカーが一意ではありません ({n}): {before[:90]}"
 script=script.replace(before,after,1)

# Phase42のActor/Critic/log_stdを全てロードする。原本はread-only。
replace_exact(
 'self.actor=torch.nn.Sequential(torch.nn.Linear(12,128),torch.nn.Tanh(),torch.nn.Linear(128,128),torch.nn.Tanh(),torch.nn.Linear(128,2))',
 'self.actor=torch.nn.Sequential(torch.nn.Linear(12,128),torch.nn.Tanh(),torch.nn.Linear(128,128),torch.nn.Tanh(),torch.nn.Linear(128,2))\n'
 '        self.critic=torch.nn.Sequential(torch.nn.Linear(12,128),torch.nn.Tanh(),torch.nn.Linear(128,128),torch.nn.Tanh(),torch.nn.Linear(128,1))\n'
 '        self.log_std=torch.nn.Parameter(torch.tensor([-0.5,-0.5]))')
replace_exact('upper.load_state_dict({k:v for k,v in state_upper.items() if k.startswith("actor.")},strict=True)',
              'upper.load_state_dict(state_upper,strict=True)')
replace_exact('evader_upper.load_state_dict({k:v for k,v in evasion_ck["actor_critic"].items() if k.startswith("actor.")},strict=True)',
              'evader_upper.load_state_dict(evasion_ck["actor_critic"],strict=True)')
replace_exact('import torch,pickle,csv,math,sys\n','import torch,pickle,csv,math,sys\nfrom torch.distributions import Normal\n')
replace_exact('rows=[]\nroles=[]','rows=[]\ntraining_rows=[]\nroles=[]')
replace_exact('with torch.no_grad():\n for step in range(1,max_steps+1):',
              'with torch.no_grad():\n for step in range(1,max_steps+1):')
replace_exact('    raw_forward=[0.0,0.0]\n',
              '    transition_pre=[]\n    raw_forward=[0.0,0.0]\n')
replace_exact('        cmd=role_policy(role_observation)[0]',
'''        # PPOで保存するactionは変換前のlatent 2D行動。
        # raw ~ Normal(actor(obs), exp(log_std)), physical=(sigmoid(raw0)*max_forward,tanh(raw1))
        dist=Normal(role_policy.actor(role_observation),role_policy.log_std.exp().expand(1,2))
        latent=dist.sample()
        old_logp=dist.log_prob(latent).sum(-1)[0]
        val=role_policy.critic(role_observation).reshape(-1)[0]
        cmd=torch.stack((torch.sigmoid(latent[0,0])*role_policy.max_forward,torch.tanh(latent[0,1])))
        assert torch.isfinite(old_logp) and torch.isfinite(val)
        transition_pre.append(dict(robot="AB"[i],step=step,role="pursuer" if pursuer else "evader",
          obs=role_observation.reshape(-1).detach().cpu().tolist(),
          action=latent.reshape(-1).detach().cpu().tolist(),
          logp=float(old_logp),value=float(val),forward=float(cmd[0]),yaw=float(cmd[1]),
          dist0=distance_now,old_yaw=float(commands[i,2])))''')
# calculate next obs from exact postphysics states and smoothed finite differences; use same observation formula
next_func=r"""
def next_observations(after_states,before_xy,before_yaws,old_vel,old_omega):
    new_xy=torch.stack([s[0][:2] for s in after_states])
    new_yaws=[]
    for s in after_states:
        w,x,y,z=s[1].unbind()
        new_yaws.append(torch.atan2(2*(w*z+x*y),1-2*(y*y+z*z)))
    new_yaws=torch.stack(new_yaws)
    vel=.8*old_vel+.2*(new_xy-before_xy)/.02
    rate=.8*old_omega+.2*torch.atan2(torch.sin(new_yaws-before_yaws),torch.cos(new_yaws-before_yaws))/.02
    result=[]
    for i in range(2):
        j=1-i
        c,sn=torch.cos(new_yaws[i]),torch.sin(new_yaws[i])
        rel=new_xy[j]-new_xy[i]
        rv=vel[j]-vel[i]
        dy=new_yaws[j]-new_yaws[i]
        speed=(vel[i]*torch.stack((c,sn))).sum()
        eight=torch.stack(((c*rel[0]+sn*rel[1])/3,(-sn*rel[0]+c*rel[1])/3,
          c*rv[0]+sn*rv[1],-sn*rv[0]+c*rv[1],
          torch.sin(dy),torch.cos(dy),speed,rate[i]))
        w,x,y,z=after_states[i][1].unbind()
        roll=torch.atan2(2*(w*x+y*z),1-2*(x*x+y*y))
        pitch=torch.asin(torch.clamp(2*(w*y-z*x),-1.,1.))
        omega=after_states[i][2]
        result.append(torch.cat((eight,torch.stack((roll,pitch,omega[0],omega[1])))))
    return result
"""
replace_exact('prev_positions=None\n',next_func+'\nprev_positions=None\n')
# for step boundary, record the pre-action velocity state before simulation
replace_exact('    scene.step()\n',
              '    snapshot_xy=current_xy.clone()\n    snapshot_yaws=current_yaws.clone()\n'
              '    snapshot_vel=prev_vel.clone()\n    snapshot_omega=prev_omega.clone()\n'
              '    scene.step()\n')
replace_exact('    actions=new.clone();last=new.clone()\n',
'''    actions=new.clone();last=new.clone()
    after_states=[state(r,indices[i]) for i,r in enumerate(robots)]
    next12=next_observations(after_states,snapshot_xy,snapshot_yaws,snapshot_vel,snapshot_omega)
    distance_after=float(torch.linalg.norm(after_states[0][0][:2]-after_states[1][0][:2]))
    # 閾値を跨ぐと次step冒頭で役割交代する。役割ごとのPPO軌跡をその手前で切る。
    role_boundary=bool((tag_armed or distance_after>=.85) and distance_after<=.80
                       and (step+1-last_tag_step)>=150)
    for i,t in enumerate(transition_pre):
        s=after_states[i]
        w,x,y,z=s[1].unbind()
        roll=torch.atan2(2*(w*x+y*z),1-2*(x*x+y*y))
        pitch=torch.asin(torch.clamp(2*(w*y-z*x),-1.,1.))
        zheight=float(s[0][2])
        fell=(zheight<.18 or abs(float(roll))>math.radians(45) or abs(float(pitch))>math.radians(45))
        progress=(t["dist0"]-distance_after) if t["role"]=="pursuer" else (distance_after-t["dist0"])
        rp=3.0*progress
        rf=.01*max(0,t["forward"])
        rpost=-.05*((max(0,abs(math.degrees(float(roll)))-10)/10)**2+
                      (max(0,abs(math.degrees(float(pitch)))-12)/10)**2)
        wr=math.hypot(float(s[2][0]),float(s[2][1]))
        rrate=-.01*max(0,wr-1.5)**2
        rchange=-.02*(t["yaw"]-t["old_yaw"])**2
        reward=rp+rf+rpost+rrate+rchange+(-5.0 if fell else 0.0)
        row=[t["step"],(step-1)*.02,step*.02,t["robot"],t["role"],
             *t["obs"],*t["action"],t["logp"],t["value"],
             t["forward"],t["yaw"],reward,rp,rf,rpost,rrate,rchange,
             int(fell),int(role_boundary),int(step==max_steps),*next12[i].detach().cpu().tolist(),
             t["dist0"],distance_after,zheight]
        assert all(math.isfinite(float(v)) for v in row[:3]+row[5:]),("nonfinite",step,i)
        training_rows.append(row)
''')
# terminal truncation is bootstrap-able; physics failure terminates
script+=r"""
header=["step","time_before_s","time_after_s","robot","role"]+["obs_"+str(i) for i in range(12)]+["action_latent_"+str(i) for i in range(2)]+[
 "old_log_prob","old_value","forward_cmd","yaw_cmd","reward","reward_progress","reward_forward","reward_posture",
 "reward_angular_rate","reward_command_change","done_fall","done_role_switch","truncated"]+[
 "next_obs_"+str(i) for i in range(12)]+["distance_before_m","distance_after_m","height_after_m"]
with (output_dir/"ppo_training_rollout.csv").open("w",newline="") as f:
 writer=csv.writer(f);writer.writerow(header);writer.writerows(training_rows)
print("PHASE46 PPO ROLLOUT ROWS",len(training_rows),"HEADER",len(header),flush=True)
"""
ast.parse(script)
assert script.index("phase45_obs12=[]")<script.index("transition_pre=[]")<script.index("role_policy=upper")
assert "role_policy.critic(role_observation)" in script
assert "dist.log_prob(latent)" in script
assert "next_observations(after_states" in script
assert "role_boundary" in script
assert "training_rows.append(row)" in script
print("PASS: Phase46 runner構文、定義順、PPO記録経路")



In [ ]:
import subprocess,sys,time,uuid,zipfile,json,hashlib
import pandas as pd,numpy as np
from pathlib import Path
from google.colab import files
hashes={}
for label,p in (("pursuit",upper_checkpoint),("evasion",evasion_checkpoint),("locomotion",checkpoints["phase112"])):
 hashes[label]=hashlib.sha256(p.read_bytes()).hexdigest()
exp=out_root/("phase46_genesis_training_rollout_"+uuid.uuid4().hex[:8])
exp.mkdir(parents=True,exist_ok=False)
out=exp/"runtime_probe";out.mkdir()
runner=out/"run_phase46.py";runner.write_text(script,encoding="utf-8")
ast.parse(runner.read_text(encoding="utf-8"))
args=[sys.executable,"-u",str(runner),"close_start__tag","20.0","0.5","80","mutual_tag",
 str(checkpoints["phase112"]),"0.0",str(upper_checkpoint),str(evasion_checkpoint),
 "asymmetric","0.45","0.0",str(cfg_paths["phase112"])]
print("Genesis 80step収集開始。モデル更新なし。")
t0=time.monotonic()
with (out/"run.log").open("w",encoding="utf-8") as f:
 result=subprocess.run(args,cwd="/content/genesis_official/examples/locomotion",stdout=f,stderr=subprocess.STDOUT)
print("終了コード",result.returncode,"経過秒",round(time.monotonic()-t0,1))
print((out/"run.log").read_text(errors="replace")[-9500:])
if result.returncode:raise RuntimeError("Genesis実行失敗。run.logを確認")
df=pd.read_csv(out/"ppo_training_rollout.csv")
expected=["obs_"+str(i) for i in range(12)]+["next_obs_"+str(i) for i in range(12)]+[
 "action_latent_0","action_latent_1","old_log_prob","old_value","reward"]
assert len(df)==160 and df.robot.value_counts().to_dict()=={"A":80,"B":80}
assert np.isfinite(df[expected].to_numpy()).all()
assert np.allclose(df.time_after_s-df.time_before_s,.02,atol=1e-6)
assert df.groupby("robot").step.nunique().eq(80).all()
for label,p in (("pursuit",upper_checkpoint),("evasion",evasion_checkpoint),("locomotion",checkpoints["phase112"])):
 assert hashlib.sha256(p.read_bytes()).hexdigest()==hashes[label],"モデル原本の変更を検出"
summary=df.groupby(["robot","role"],as_index=False).agg(count=("step","size"),reward_mean=("reward","mean"),
  falls=("done_fall","sum"),role_boundaries=("done_role_switch","sum"))
print("実行結果：");display(summary)
summary.to_csv(out/"summary_ja.csv",index=False,encoding="utf-8-sig")
(out/"manifest.json").write_text(json.dumps({"phase":"46_training_rollout","steps":80,"samples":len(df),
 "genesis":True,"model_updated":False,"actor_latent_action":True,"old_log_prob":True,
 "critic_value":True,"next_obs":True,"role_switch_boundary":True,
 "truncated_bootstrap_required":True,"checkpoint_sha256_before_after":hashes,
 "note":"学習用ロールアウトの収集のみ。PPOの更新・性能改善は未実施。"},ensure_ascii=False,indent=2),encoding="utf-8")
archive=exp.parent/(exp.name+"_results.zip")
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
 for f in out.rglob("*"):
  if f.is_file():z.write(f,f.relative_to(exp))
print("結果ZIP",archive)
files.download(str(archive))

